# Capítulo 5 — Drawdowns

**Objetivo:** medir caídas desde máximos históricos (*underwater chart*).

### Fórmula
$$DD_t = \frac{V_t}{\max_{s\le t} V_s} - 1$$

El **Maximum Drawdown** es $\min_t DD_t$.


### Serie de drawdown

In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
import portfolio_utils as pu

daily = pd.read_csv(ROOT/"data"/"precios_ajustados_diarios.csv", index_col=0, parse_dates=True)
monthly = pu.to_month_end(daily).loc[: "2026-09-30"]
bt = pu.backtest_portfolio(monthly, pu.CURRENT_WEIGHTS, start="2016-01-31", end="2026-09-30", rebalance="A")
spy = pu.backtest_portfolio(monthly, {"SPY":1}, start="2016-01-31", end="2026-09-30", rebalance="N")

dd_p = pu.drawdown_series(bt["value"])
dd_s = pu.drawdown_series(spy["value"])
print("Max DD portafolio:", f"{dd_p.min()*100:.2f}%")
print("Max DD SPY:", f"{dd_s.min()*100:.2f}%")


Max DD portafolio: -27.47%
Max DD SPY: -23.93%


### Gráfico underwater

In [2]:
fig, ax = plt.subplots(figsize=(11,5))
ax.fill_between(dd_p.index, dd_p.values*100, 0, alpha=0.45, label="Portafolio")
ax.plot(dd_s.index, dd_s.values*100, label="SPY", color="C1")
ax.set_ylabel("Drawdown (%)")
ax.set_title("Drawdowns — portafolio actual vs SPY")
ax.legend(); plt.tight_layout()
plt.savefig(ROOT/"figures"/"03_drawdowns.png", dpi=120)
plt.show()


![figura](../figures/03_drawdowns.svg)


**Interpretación:** el drawdown de 2022 es el evento dominante (PDF: ene–sep 2022, −25.6%). Con más `SMH` e `IEMG`, el Max DD actualizado es algo más profundo que el del PDF.

## Conclusiones
- Superar a SPY en CAGR no implica mejores peores escenarios.
- Tiempo bajo el agua importa tanto como la profundidad.

## Ejercicios propuestos
1. Calcula la duración (meses) del peores drawdown.
2. Compara Max DD con rebalanceo mensual vs anual.
3. ¿Qué activo individual tiene el peor Max DD del núcleo?
